# 7. ReAct Agent from Scratch
**Industry:** Supply Chain

Implement a ReAct (Reason + Act) agent using LangGraph that answers questions by reasoning step-by-step and calling an external tool.

In [1]:
!pip install langgraph langchain langchain-openai requests


[notice] A new release of pip is available: 24.2 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import json
import requests
from typing import Annotated, TypedDict
from langchain_core.messages import SystemMessage, HumanMessage, ToolMessage, AIMessage
import os
import dotenv
dotenv.load_dotenv(r"D:/Internship/Teach-ai/Backend/.env")
from langchain_openai import AzureChatOpenAI
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
import os

@tool
def tracking_api(shipment_id: str) -> str:
    """Get the current status and location of a shipment by ID using a real public API."""
    # Using a public ZipCode API as a mock for "Logistics location"
    # e.g., if shipment_id is a zip code like 90210, it tells us where the package is.
    if len(shipment_id) != 5 or not shipment_id.isdigit():
        return "Error: Please provide a 5-digit zip code as the shipment ID to track its location."
    
    response = requests.get(f"https://api.zippopotam.us/us/{shipment_id}")
    if response.status_code == 200:
        data = response.json()
        place = data['places'][0]
        return f"Shipment {shipment_id} is currently at a facility in {place['place name']}, {place['state']}. Status: In Transit."
    else:
        return "Shipment not found or tracking API is down."

tools = [tracking_api]
llm = AzureChatOpenAI(azure_endpoint=os.environ.get("AZURE_OPENAI_ENDPOINT"), api_key=os.environ.get("AZURE_OPENAI_API_KEY"), azure_deployment="gpt-4o", api_version=os.environ.get("AZURE_OPENAI_API_VERSION", "2024-12-01-preview")).bind_tools(tools)

class AgentState(TypedDict):
    messages: Annotated[list, add_messages]

def reason(state: AgentState):
    response = llm.invoke(state["messages"])
    return {"messages": [response]}

def should_continue(state: AgentState):
    last_message = state["messages"][-1]
    if last_message.tool_calls:
        return "tools"
    return END

workflow = StateGraph(AgentState)
workflow.add_node("reason", reason)
workflow.add_node("tools", ToolNode(tools))

workflow.add_edge(START, "reason")
workflow.add_conditional_edges("reason", should_continue, ["tools", END])
workflow.add_edge("tools", "reason")

app = workflow.compile()

query = "Where is shipment #90210 right now and what state is it in?"
for event in app.stream({"messages": [HumanMessage(content=query)]}):
    for node, value in event.items():
        msg = value["messages"][-1]
        if isinstance(msg, AIMessage) and msg.tool_calls:
            print(f"\n[Reasoning -> Action]: Thinking about calling tool {msg.tool_calls[0]['name']} with args {msg.tool_calls[0]['args']}")
        elif isinstance(msg, ToolMessage):
            print(f"\n[Observation]: {msg.content}")
        elif isinstance(msg, AIMessage):
            print(f"\n[Final Answer]: {msg.content}")


[Reasoning -> Action]: Thinking about calling tool tracking_api with args {'shipment_id': '90210'}



[Observation]: Shipment 90210 is currently at a facility in Beverly Hills, California. Status: In Transit.



[Final Answer]: Shipment #90210 is currently at a facility in Beverly Hills, California, and its status is "In Transit."
